In [64]:
import itertools
from pathlib import Path
import re

import matplotlib.pyplot as plt
import mne
from mne.decoding import ReceptiveField
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, KFold
from scipy.stats import ttest_ind
import seaborn as sns
from tqdm.auto import tqdm

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
outdir = "."

In [7]:
all_epoch_paths = list(Path("epochs").glob("*.fif"))

In [10]:
epochs = {}
for path in tqdm(all_epoch_paths):
    subject_name = re.findall("(EC[\d]+)_epochs", str(path))[0]
    if subject_name == "EC282":
        # missing ecog data
        continue
    epochs[subject_name] = mne.read_epochs(str(path)).pick("ecog").resample(100)

    # DEV
    break

  0%|          | 0/9 [00:00<?, ?it/s]

Reading /userdata/jgauthier/projects/barakeet/epochs/EC248_epochs.fif ...


/tmp/ipykernel_112794/1425572251.py:7: RuntimeWarning: This filename (epochs/EC248_epochs.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz
  epochs[subject_name] = mne.read_epochs(str(path)).pick("ecog").resample(100)


Isotrak not found
    Found the data of interest:
        t =    -200.00 ...    1000.00 ms
        0 CTF compensation matrices available
Adding metadata with 71 columns
864 matching events found
No baseline correction applied
0 projection items activated


In [33]:
md

,wav_file,stim_number,word_end,non_word,phoneme_pair,morph_n,base,file_format,root,word_side,...,frameRate,Unnamed: 65,binned_responses,acoustic_slider,slider_reorient,lexical_evidence,mismatch,linear_acoustics,linear_acoustic_cue,categorical_acoustic_cue
0,stimuli/35_penecillin_pb_010.wav,35.0,penecillin,benecillin,pb,10.0,35_penecillin_pb_010,wav,stimuli/,right,...,59.685618,None,2,7.9250,3.0750,0,0.8,0.6,0.6,1
1,stimuli/2_bountiful_bm_005.wav,2.0,bountiful,mountiful,bm,5.0,2_bountiful_bm_005,wav,stimuli/,left,...,59.685618,None,3,8.0625,8.0625,0,0.8,0.6,0.6,1
2,stimuli/11_desolate_dn_002.wav,11.0,desolate,nesolate,dn,2.0,11_desolate_dn_002,wav,stimuli/,right,...,59.685618,None,2,3.0375,3.0375,0,0.0,-1.0,-1.0,-1
3,stimuli/35_beneficial_pb_006.wav,35.0,beneficial,peneficial,pb,6.0,35_beneficial_pb_006,wav,stimuli/,right,...,59.685618,None,3,9.9000,1.1000,1,1.0,-1.0,-1.0,-1
4,stimuli/11_desolate_dn_004.wav,11.0,desolate,nesolate,dn,4.0,11_desolate_dn_004,wav,stimuli/,right,...,59.685618,None,2,3.0750,3.0750,0,0.4,-0.2,-0.2,-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
859,stimuli/35_penecillin_pb_009.wav,35.0,penecillin,benecillin,pb,9.0,35_penecillin_pb_009,wav,stimuli/,left,...,59.432070,None,3,8.5125,2.4875,0,0.6,0.2,0.2,1
860,stimuli/35_beneficial_pb_011.wav,35.0,beneficial,peneficial,pb,11.0,35_beneficial_pb_011,wav,stimuli/,left,...,59.432070,None,2,3.1375,7.8625,1,0.0,1.0,1.0,1
861,stimuli/11_necessary_dn_006.wav,11.0,necessary,decessary,dn,6.0,11_necessary_dn_006,wav,stimuli/,left,...,59.432070,None,3,8.4125,8.4125,1,0.2,0.6,0.6,1
862,stimuli/35_penecillin_pb_008.wav,35.0,penecillin,benecillin,pb,8.0,35_penecillin_pb_008,wav,stimuli/,left,...,59.432070,None,3,8.2125,2.7875,0,0.4,-0.2,-0.2,-1


In [53]:
ep = epochs["EC248"]
md = ep.metadata

assert set(md.resampled) == set(range(1, int(md.resampled.max()) + 1))

# Prepare regression features

# linear acoustic cue: `resampled` centered and scaled to [-1, 1]
md["linear_acoustic_cue"] = (md.resampled - np.mean(list(set(md.resampled)))) / (md.resampled.max() - md.resampled.min()) * 2
assert np.isclose(md.linear_acoustics.min(), -1)
assert np.isclose(md.linear_acoustics.max(), 1)
assert np.isclose(md.linear_acoustics.mean(), 0)  # true if data is balanced
for phoneme_pair, group in md.groupby("phoneme_pair"):
    assert np.isclose(group.linear_acoustic_cue.min(), -1)
    assert np.isclose(group.linear_acoustic_cue.max(), 1)
    assert np.isclose(group.linear_acoustic_cue.mean(), 0)  # true if data is balanced

# categorical acoustic cue: mapped to {-1, 1}; 1 = resampled > 0
md["categorical_acoustic_cue"] = (md.linear_acoustic_cue > 0).astype(int) * 2 - 1
assert md.categorical_acoustic_cue.mean() == 0
for phoneme_pair, group in md.groupby("phoneme_pair"):
    assert group.categorical_acoustic_cue.min() == -1
    assert group.categorical_acoustic_cue.max() == 1
    assert group.categorical_acoustic_cue.mean() == 0

# TODO more features

In [82]:
# Prepare design matrix
features_per_phoneme_pair = [
    ("linear_acoustic_cue", "onset"),
    ("categorical_acoustic_cue", "onset"),
]

X, Y = [], []
phoneme_pairs = sorted(set(md.phoneme_pair))
feature_names = [f"{feature_name}-{phoneme_pair}"
                 for feature_name, _ in features_per_phoneme_pair
                 for phoneme_pair in phoneme_pairs]
for idx, ep_i in enumerate(ep):
    # ep_i: n_channels * n_samples
    md_i = md.iloc[idx]

    # build up design matrix for this trial by column
    Xi = []
    for feature_name, feature_alignment in features:
        for phoneme_pair in phoneme_pairs:
            Xij = np.zeros((ep_i.shape[1], 1))
            if phoneme_pair == md_i.phoneme_pair:
                if feature_alignment == "onset":
                    onset_time = 0. - ep.tmin
                    onset_sample = int(onset_time * ep.info["sfreq"])
                    Xij[onset_sample] = md_i[feature_name]
                else:
                    raise ValueError(f"Unknown feature alignment: {feature_alignment}")
            Xi.append(Xij)
    Xi = np.concatenate(Xi, axis=1)
    X.append(Xi)

    Y.append(ep_i)

In [83]:
X = np.concatenate(X, axis=0)
Y = np.concatenate(Y, axis=1).T
X.shape, Y.shape

((103680, 6), (103680, 384))

In [84]:
model = ReceptiveField(tmin=-0.1, tmax=0.7, sfreq=ep.info["sfreq"],
                       feature_names=feature_names)

scores = []
# TODO stratify the split
for train, test in KFold(n_splits=3, shuffle=False).split(X, Y):
    model.fit(X[train], Y[train])
    scores.append(model.score(X[test], Y[test]))

Fitting 1 epochs, 6 channels


  0%|          | Sample : 0/27 [00:00<?,       ?it/s]

Fitting 1 epochs, 6 channels


  0%|          | Sample : 0/27 [00:00<?,       ?it/s]

Fitting 1 epochs, 6 channels


  0%|          | Sample : 0/27 [00:00<?,       ?it/s]

In [85]:
np.stack(scores).mean(axis=0).max()

0.012259703915049084